# SEC-Filing Q&A — grounded, traceable, honest

A lightweight agentic prototype that answers natural-language questions about
SEC filings (10-K / 10-Q) sourced from **PDFs**.

The prior system at the client lost trust by **hallucinating values, mixing
annual/quarterly numbers, confusing GAAP vs non-GAAP, and giving no
traceability**. This prototype is built around one principle that kills all
four:

> **The LLM handles language; deterministic code handles every number; and
> every number must appear verbatim in the source filing before it can enter an
> answer.**

```
   LLM reads the QUESTION  ->  CODE finds the NUMBER (+proof)  ->  LLM writes the SENTENCE
                             == deterministic numeric spine over THE BANK ==
```

This notebook is a live walkthrough: build the Bank, ask numeric and prose
questions, show the trace behind every answer, and demonstrate the trust
behaviors (disambiguation, honest refusal) plus a hand-verified benchmark.


## 1. Setup

All modules are plain Python in this project. The LLM layer is isolated in
`llm.py`; if no API key / credits are configured the pipeline runs on a
deterministic fallback (keyword understanding + extractive prose), so this
notebook executes end-to-end regardless.

In [1]:
import config, edgar, extract, prose, bank, core, benchmark, llm
import pandas as pd, textwrap

print("model:", config.ANTHROPIC_MODEL, "(escalate:", config.ANTHROPIC_MODEL_ESCALATE + ")")
print("live LLM available:", llm.available(),
      "-> language steps use", "the live model" if llm.available() else "the deterministic fallback")

model: claude-haiku-4-5 (escalate: claude-sonnet-5-5)
live LLM available: True -> language steps use the live model


## 2. The Bank (system of record)

Ingestion happens **once**: we render the EDGAR HTML filing to PDF (on purpose —
the hard problem is messy PDF tables, not HTML `<table>` tags), then parse it
into two stores:

- **Fact Store** — structured statement values, each with provenance
  (filing, page, raw cell) and a grounding flag.
- **Prose Store** — Risk Factors / MD&A passages with page provenance.

The answer path only ever **reads** the Bank. Below we build it from the already
rendered Tesla FY2025 10-K.

In [2]:
ent = edgar.resolve_entity("Tesla")[0][0]
filing = {"form": "10-K", "accession_no": "0001628280-25-000000"}
pdf = config.FILINGS_DIR / "TSLA_10K_2025-12-31.pdf"

if not bank.load_facts():
    bank.write_facts(extract.facts_to_dicts(extract.extract_facts(pdf, ent, filing)))
if not bank.load_prose():
    bank.write_prose(prose.extract_prose(pdf, ent, filing))

facts = bank.load_facts()
prose_chunks = bank.load_prose()
print(f"entity: {ent.title}  (CIK {ent.cik}, {ent.ticker})")
print(f"Fact Store : {len(facts)} facts   | all grounded: {all(f['grounding_ok'] for f in facts)}")
print(f"Prose Store: {len(prose_chunks)} passages")
print("periods available:", bank.periods_for(ent.cik))

entity: Tesla, Inc.  (CIK 1318605, TSLA)
Fact Store : 268 facts   | all grounded: True
Prose Store: 190 passages
periods available: ['ASOF2024-12-31', 'ASOF2025-12-31', 'FY2023', 'FY2024', 'FY2025']


### What a grounded fact looks like

Every number carries the fields a skeptic needs to verify it: the raw label as
printed, the period, the unit/scale, the page, the exact printed cell text, and
`grounding_ok` (the cell appears verbatim on that page).

In [3]:
df = pd.DataFrame(facts)
keep = ["Total revenues", "Income from operations",
        "Net income attributable to common stockholders",
        "Cash and cash equivalents"]
view = df[df.raw_label.isin(keep)][
    ["statement","raw_label","period","raw_cell_text","unit","scale","page_no","grounding_ok"]]
view.sort_values(["raw_label","period"]).reset_index(drop=True)

,statement,raw_label,period,raw_cell_text,unit,scale,page_no,grounding_ok
0,balance_sheet,Cash and cash equivalents,ASOF2024-12-31,"16,139",USD,millions,80,True
1,balance_sheet,Cash and cash equivalents,ASOF2025-12-31,"16,513",USD,millions,80,True
2,operations,Income from operations,FY2023,"8,891",USD,millions,82,True
3,operations,Income from operations,FY2024,"7,076",USD,millions,82,True
4,operations,Income from operations,FY2025,"4,355",USD,millions,82,True
5,operations,Net income attributable to common stockholders,FY2023,"14,997",USD,millions,82,True
6,operations,Net income attributable to common stockholders,FY2024,"7,091",USD,millions,82,True
7,operations,Net income attributable to common stockholders,FY2025,"3,794",USD,millions,82,True
8,operations,Total revenues,FY2023,"96,773",USD,millions,82,True
9,operations,Total revenues,FY2024,"97,690",USD,millions,82,True


### The disambiguation hazard, handled in data

`Basic` / `Diluted` appear **twice** in the income statement — once as earnings
per share ($/share) and once as weighted-average share counts (millions). Same
label, different meaning. The extractor keeps them apart using the section
header, so a question for EPS can never silently return a share count.

In [4]:
amb = df[df.raw_label.isin(["Basic","Diluted"])][
    ["raw_label","period","raw_cell_text","unit","scale","section"]]
amb.assign(section=amb.section.str.slice(0,42)+"...").reset_index(drop=True)

,raw_label,period,raw_cell_text,unit,scale,section
0,Basic,FY2025,1.18,USD,units,Net income per share of common stock attri...
1,Basic,FY2024,2.23,USD,units,Net income per share of common stock attri...
2,Basic,FY2023,4.73,USD,units,Net income per share of common stock attri...
3,Diluted,FY2025,1.08,USD,units,Net income per share of common stock attri...
4,Diluted,FY2024,2.04,USD,units,Net income per share of common stock attri...
5,Diluted,FY2023,4.30,USD,units,Net income per share of common stock attri...
6,Basic,FY2025,"3,225",shares,millions,Weighted average shares used in computing ...
7,Basic,FY2024,"3,197",shares,millions,Weighted average shares used in computing ...
8,Basic,FY2023,"3,174",shares,millions,Weighted average shares used in computing ...
9,Diluted,FY2025,"3,528",shares,millions,Weighted average shares used in computing ...


## 3. Numeric Q&A — answer + full trace

`core.answer()` runs the agentic flow: understand -> resolve entity -> find the
number in the Bank -> compute -> confidence gate -> phrase. The helper below
prints the answer plus the trace (reasoning steps + citations) behind it.

In [5]:
def show(q):
    a = core.answer(q)
    badge = {"answered": "ANSWERED", "hedged": "HEDGED", "refused": "REFUSED"}[a.status]
    print("Q:", q)
    print(f"[{badge} | confidence {a.confidence}]")
    print(textwrap.fill(a.text, 92))
    for s in a.trace.steps:
        print("   ·", s)
    for cc in a.trace.citations:
        line = f"   ↳ {cc.get('form') or '10-K'} p{cc['page']} — {cc['label']}"
        if cc.get("period"):
            line += f" [{cc['period']}] = {cc['cell']} ({cc['unit']}/{cc['scale']})"
        if cc.get("score") is not None:
            line += f"  (match {cc['score']})"
        print(line)
        if cc.get("snippet"):
            print("       ", textwrap.fill(cc["snippet"], 86, subsequent_indent="        "))
    print()
    return a

In [6]:
_ = show("What was Tesla's total revenue in 2025?")
_ = show("What was Tesla's net income in 2025?")
_ = show("What was Tesla's diluted EPS in 2025?")
_ = show("How much cash did Tesla have at the end of 2025?")

Q: What was Tesla's total revenue in 2025?
[ANSWERED | confidence 0.98]
Tesla's total revenue in 2025 was $94,827 million.
   · understood: metric=revenue years=[2025] calc=none kind=numeric company='Tesla'
   · resolved entity: Tesla, Inc. (CIK 1318605)
   · matched 'Total revenues' FY2025 = 94,827 (p82)
   ↳ 10-K p82 — Total revenues [FY2025] = 94,827 (USD/millions)



Q: What was Tesla's net income in 2025?
[ANSWERED | confidence 0.98]
Tesla's net income attributable to common stockholders in 2025 was $3,794 million.
   · understood: metric=net_income years=[2025] calc=none kind=numeric company='Tesla'
   · resolved entity: Tesla, Inc. (CIK 1318605)
   · matched 'Net income attributable to common stockholders' FY2025 = 3,794 (p82)
   ↳ 10-K p82 — Net income attributable to common stockholders [FY2025] = 3,794 (USD/millions)



Q: What was Tesla's diluted EPS in 2025?
[ANSWERED | confidence 0.98]
Tesla, Inc.'s diluted EPS in 2025 was $1.08.
   · understood: metric=eps_diluted years=[2025] calc=none kind=numeric company='Tesla'
   · resolved entity: Tesla, Inc. (CIK 1318605)
   · matched 'Diluted' FY2025 = 1.08 (p82)
   ↳ 10-K p82 — Diluted [FY2025] = 1.08 (USD/units)



Q: How much cash did Tesla have at the end of 2025?
[ANSWERED | confidence 0.98]
At the end of 2025, Tesla had $16,513 million in cash and cash equivalents.
   · understood: metric=cash years=[2025] calc=none kind=numeric company='Tesla'
   · resolved entity: Tesla, Inc. (CIK 1318605)
   · matched 'Cash and cash equivalents' ASOF2025-12-31 = 16,513 (p80)
   ↳ 10-K p80 — Cash and cash equivalents [ASOF2025-12-31] = 16,513 (USD/millions)



Note the net-income answer: it returns the figure *attributable to common
stockholders* (the EPS numerator) **and flags** that a total "Net income" line
also exists — exactly the GAAP-subtotal confusion the prior system fell for.

### Computations are done in code, never by the LLM

In [7]:
_ = show("How did Tesla's revenue change from 2024 to 2025?")
_ = show("What was Tesla's operating margin in 2025?")

Q: How did Tesla's revenue change from 2024 to 2025?
[ANSWERED | confidence 0.98]
Tesla's revenue decreased from $97,690 million in 2024 to $94,827 million in 2025,
representing a decline of 2.9%.
   · understood: metric=revenue years=[2024, 2025] calc=growth kind=numeric company='Tesla'
   · resolved entity: Tesla, Inc. (CIK 1318605)
   · growth = (94,827 - 97,690) / 97,690 = -2.9%
   ↳ 10-K p82 — Total revenues [FY2024] = 97,690 (USD/millions)
   ↳ 10-K p82 — Total revenues [FY2025] = 94,827 (USD/millions)



Q: What was Tesla's operating margin in 2025?
[ANSWERED | confidence 0.98]
Tesla's operating margin in 2025 was 4.6%, calculated from operating income of $4,355
million divided by total revenue of $94,827 million.
   · understood: metric=operating_income years=[2025] calc=margin kind=numeric company='Tesla'
   · resolved entity: Tesla, Inc. (CIK 1318605)
   · margin = 4,355 / 94,827 = 4.6%
   ↳ 10-K p82 — Income from operations [FY2025] = 4,355 (USD/millions)
   ↳ 10-K p82 — Total revenues [FY2025] = 94,827 (USD/millions)



## 4. Trust behaviors — refusing beats fabricating

A correct "I don't have that" is worth more than a confident wrong number. The
system refuses when the value isn't in the Bank and when it can't identify the
company, instead of guessing.

In [8]:
_ = show("What was Tesla's net income in 2019?")       # not in the banked filings
_ = show("What does management say about gross margin?")  # no company named

Q: What was Tesla's net income in 2019?
[REFUSED | confidence 0.0]
I don't have Net income attributable to common stockholders for 2019 in the banked filings
for Tesla, Inc., so I won't fabricate it.
   · understood: metric=net_income years=[2019] calc=none kind=numeric company='Tesla'
   · resolved entity: Tesla, Inc. (CIK 1318605)



Q: What does management say about gross margin?
[REFUSED | confidence 0.0]
I couldn't identify the company in that question, so I won't guess. Please name the company
or ticker.
   · understood: metric=None years=[] calc=none kind=prose company=''



## 5. Prose Q&A — grounded narration with sources

Qualitative questions are answered from the Risk Factors / MD&A text of the same
filing. The system retrieves relevant passages, narrates **only over them**, and
**ships the page-cited source passages with the answer** so you can verify it.
(With live credits the LLM narrates; offline it shows the top retrieved passage
verbatim — labeled as such.)

In [9]:
_ = show("What are the main risks Tesla highlights about competition and demand?")
_ = show("What does Tesla's MD&A say about liquidity and capital resources?")

Q: What are the main risks Tesla highlights about competition and demand?
[ANSWERED | confidence 0.29]
Based on the excerpts, Tesla highlights two main competitive and demand risks:  **Demand
Risk**: Tesla warns that "if the market for electric vehicles in general and Tesla vehicles
in particular does not develop as we expect, develops more slowly than we expect, or if
demand for our vehicles decreases in our markets or our vehicles compete with each other,
our business, prospects, financial condition and operating results may be harmed" [p.28].
The company identifies specific demand concerns including consumer perceptions about EV
features, quality, safety, performance, cost, limited range, and charging access [p.28].
**Competition Risk**: Tesla faces "competition and uncertainty regarding the future of
electric vehicles, autonomous solutions or our other products and services" [p.32],
including competition from alternative fuel vehicles and plug-in hybrids [p.28].
Additionally, the c

Q: What does Tesla's MD&A say about liquidity and capital resources?
[ANSWERED | confidence 0.29]
Based on Tesla's MD&A, the company states that it "generally expect[s] to continue to
generate net positive operating cash flow" and believes "that our current sources of funds
will provide us with adequate liquidity during the 12-month period" [p.70]. The company can
adjust its capital and operating expenditures by segment and may raise additional capital
through debt facilities or alternative financing sources to fund business growth, or
conversely repay indebtedness early [p.70]. However, the MD&A also notes that Tesla's
business is "capital-intensive" with fluctuating cash timing, and the company "may need or
want to raise additional funds through the issuance of equity, equity-related or debt
securities" to support expansion and strategic investments [p.37].
   · understood: metric=None years=[] calc=none kind=prose company='Tesla'
   · resolved entity: Tesla, Inc. (CIK 1318605)
   · 

## 6. Benchmark — the objective trust check

A small hand-verified set: each value was read by hand from the FY2025 10-K; the
engine must land within a tolerance band and must **refuse** the out-of-corpus
question. This is what lets us *claim* trustworthiness rather than assert it.

In [10]:
_ = benchmark.run()

[PASS] What was Tesla's total revenue in 2025?
       status=answered (exp answered)  value=94,827.000 (exp 94,827.000)


[PASS] What was Tesla's net income in 2025?
       status=answered (exp answered)  value=3,794.000 (exp 3,794.000)


[PASS] What was Tesla's operating income in 2025?
       status=answered (exp answered)  value=4,355.000 (exp 4,355.000)


[PASS] What was Tesla's diluted EPS in 2025?
       status=answered (exp answered)  value=1.080 (exp 1.080)


[PASS] How much cash did Tesla have at the end of 2025?
       status=answered (exp answered)  value=16,513.000 (exp 16,513.000)


[PASS] How did Tesla's revenue change from 2024 to 2025?
       status=answered (exp answered)  value=-2.931 (exp -2.931)


[PASS] What was Tesla's operating margin in 2025?
       status=answered (exp answered)  value=4.593 (exp 4.593)


[PASS] What was Tesla's net income in 2019?
       status=refused (exp refused)  value=- (exp -)

8/8 cases passed


There is also a hermetic unit-test suite (no network / no live API):

```bash
python3 -m pytest tests/ -q      # 159 tests
```

covering number parsing, grounding (incl. the failure branch), label/scale/
period disambiguation, the confidence gate, and the prose retriever.

## 7. What changes at scale (and what doesn't)

**Demo scale:** 1 company, 1 filing, ~160 facts + ~190 prose chunks — JSON on
disk, in-memory matching.

**At tens of thousands of filings:**

| Layer | Prototype | At scale |
|---|---|---|
| Fact Store | JSON + in-memory filter | columnar store (DuckDB/Parquet), partitioned by company/period + index |
| Ingestion | parse one PDF inline | batch pipeline; messy tables escalate to an LLM/vision fallback (still grounding-checked) |
| Prose retrieval | TF-IDF cosine, in-memory | dense embeddings + a vector index — same `ProseIndex` interface |
| EDGAR | sparing per-filing fetch | bulk-archive ingestion; EDGAR stays a constrained resource, never a per-query scrape |

**What does not change:** the query layer and the trust invariants — grounding,
disambiguation, the confidence gate, and traceability — because the answer path
only reads the Bank.

### Honest weaknesses
Table extraction on messy layouts (pdfplumber/Camelot ~65–75% on complex tables
— the #1 thing that breaks at full scale); label disambiguation; restated /
amended filings; period-alignment edge cases; and prose answers are *paraphrased*
(softer grounding than verbatim numbers), with lexical retrieval liable to miss a
paraphrased query a dense embedder would catch.